[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn
import math

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

class GroupQueryAttention:
    def __init__(self, d_model, num_heads, num_kv_heads):
        # pass  # Initialize projections
        # 每个KV head 对应 num_heads // num_kv_heads 个 Q head
        assert num_heads % num_kv_heads == 0
        assert d_model % num_heads == 0 
        self.nkvhead=num_kv_heads
        self.nhead = num_heads
        self.hdim = d_model//self.nhead
        self.W_q = nn.Linear(d_model, d_model)  # (d_model, d_model)
        self.W_k = nn.Linear(d_model, self.nkvhead*self.hdim)
        self.W_v = nn.Linear(d_model, self.nkvhead*self.hdim)

        self.W_o = nn.Linear(d_model, d_model)

        

    def forward(self, x):
        q, k, v = self.W_q(x), self.W_k(x), self.W_v(x)
        seq_q = q.shape[1]
        bz, seq_k, d_model = k.shape
        dim = self.hdim
        repeat_ratio = self.nhead // self.nkvhead
        q = q.reshape(bz, seq_q, self.nhead, dim).transpose(1,2) 
        k = k.reshape(bz, seq_k, self.nkvhead, dim).transpose(1,2).repeat(1,repeat_ratio ,1,1)
        v = v.reshape(bz, seq_k, self.nkvhead, dim).transpose(1,2).repeat(1,repeat_ratio ,1,1)
        attn_w = q @ k.transpose(-1,-2) # bz nhead sq sk
        attn_w = attn_w * (dim ** -0.5) # 缩放
        attn_w = torch.softmax(attn_w, dim=-1) # softamx
        out = (attn_w @ v).transpose(1,2).reshape(bz, seq_q, -1)
        out = self.W_o(out)
        return out

        # pass  # Self-attention with grouped KV

In [9]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [10]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (2.8ms)
  ✅ [2/5] nn.Linear with correct shapes (0.5ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (1.1ms)
  ✅ [4/5] KV heads are shared correctly (3.5ms)
  ✅ [5/5] Gradient flow (43.2ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (51.1ms total)
  Progress saved. Run status() to see your dashboard.

